# Adversarial robustness — SWIFT payments

Build a binary classifier (URGENCY vs not) on synthetic SWIFT transaction data from the obfuscation-study generator, then attack it with an FGSM-style gradient perturbation and measure robust accuracy / ASR across perturbation budgets.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import pandas as pd
from examples.adversarial import swift_binary_dataset, train_test, train, robustness_sweep
from mcp_servers import robustness_tools as rt

X, y, feats, target = swift_binary_dataset(2000, seed=42)
Xtr, Xte, ytr, yte = train_test(X, y)
print('features:', feats, '| target:', target)
print('train/test:', Xtr.shape[0], '/', Xte.shape[0])

In [ ]:
model = train('lr', Xtr, ytr)
clean = model.predict(Xte)
print('clean accuracy: %.3f' % (clean == yte).mean())
print(rt.adversarial_robustness_checklist('sklearn', 'tabular', high_stakes=False))

In [ ]:
epsilons = [0.0, 0.1, 0.25, 0.5, 1.0, 2.0]
rows = robustness_sweep(model, Xte, yte, epsilons)
df = pd.DataFrame(rows)
print(df.to_string(index=False))

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(df['eps'], df['clean_accuracy'], 'o-', label='clean accuracy', color='#4f8cff')
ax.plot(df['eps'], df['robust_accuracy'], 'o-', label='robust accuracy', color='#e05b5b')
ax.plot(df['eps'], df['asr'], 'o--', label='ASR (on correct)', color='#d9a441')
ax.set_xlabel('perturbation budget eps'); ax.set_ylabel('accuracy')
ax.legend(); ax.set_title('SWIFT classifier — robustness vs eps')
ax.grid(alpha=0.3)